# Phases 4–6: RGB vs HSV experiment

This notebook does three phases of the capstone:
- **Phase 4:** gets your plant photos ready (load, split, resize, augment, colour-convert)
- **Phase 5:** trains the RGB models
- **Phase 6:** trains the identical HSV models

**Before you start:** Runtime → Change runtime type → **GPU**.

**How to run it:** top to bottom, one cell at a time (Shift+Enter). Read the short note above each cell first. It says what the cell does and what you should see.

**If Colab disconnects:** run everything from the top again. Every finished training run is saved to your Google Drive and skipped automatically, so finished work is never lost.

## Setup

### Step 1: Load the toolboxes and connect Google Drive
Same idea as Phase 1: `import` loads toolboxes. `drive.mount` connects your Drive (approve the pop-up), so the notebook can read your photos and save results somewhere permanent. Colab's own storage is wiped every time it disconnects.

**Expected output:** "Mounted at /content/drive", a TensorFlow version, and a GPU listed.

In [ ]:
import os, time, shutil, zipfile, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

from google.colab import drive
drive.mount('/content/drive')

print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

### Step 2: Experiment settings
Every number that controls the experiment lives in this one cell. The key idea of your whole project: **RGB and HSV runs use exactly these same settings.** The colour space is the only thing that changes.

- `SPLIT_SEED` fixes which photos go into train / validation / test, so every run uses the identical split.
- `RUN_SEEDS`: each model is trained **3 times** from different random starting points. If HSV beats RGB in all 3, that's a real effect. If it wins 1 out of 3, it's probably luck.
- `ARCHITECTURES`: two kinds of model, explained in Phase 5.
- `EPOCHS`: how many times each model goes through the whole training pile. Same for RGB and HSV.

**Expected output:** nothing. This cell just stores settings.

In [ ]:
DRIVE_FOLDER = '/content/drive/MyDrive/plant_health_capstone'
DRIVE_DATASET = f'{DRIVE_FOLDER}/plantvillage/PlantVillage'  # your photos in Drive
DRIVE_ZIP = f'{DRIVE_FOLDER}/plantvillage_dataset.zip'  # one-file copy, made automatically
LOCAL_DATA = '/content/data'  # fast local copy used for training
RESULTS_DIR = f'{DRIVE_FOLDER}/results'  # everything you'll need for the paper

IMG_SIZE = 224  # every photo is resized to 224 x 224 pixels
BATCH_SIZE = 32  # photos the model looks at per training step
SPLIT_SEED = 123  # fixes the train / validation / test split
RUN_SEEDS = [1, 2, 3]  # each model is trained 3 times

ARCHITECTURES = ['mobilenet', 'scratch']
EPOCHS = {'mobilenet': 10, 'scratch': 15}

os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS_CSV = f'{RESULTS_DIR}/results.csv'
AUTOTUNE = tf.data.AUTOTUNE

# Phase 4: The data pipeline

### Step 3: Copy the photos onto Colab's local disk
Reading 20,000 separate photos straight from Google Drive is very slow. You saw this when moving them took 16 minutes. So this cell does two things:

1. **First time only:** bundles all the photos into **one zip file** in your Drive. This part can take 10–20 minutes. Let it finish.
2. **Every session:** copies that single zip to Colab and unzips it. This takes about a minute.

**Expected output:** "Dataset ready at /content/data".

In [ ]:
if not os.path.exists(DRIVE_ZIP):
    print('First time only: zipping the dataset in Drive. This can take 10-20 minutes...')
    shutil.make_archive('/content/plantvillage_dataset', 'zip', root_dir=DRIVE_DATASET)
    shutil.copy('/content/plantvillage_dataset.zip', DRIVE_ZIP)

if not os.path.exists(LOCAL_DATA):
    if not os.path.exists('/content/plantvillage_dataset.zip'):
        print('Copying the zip from Drive...')
        shutil.copy(DRIVE_ZIP, '/content/plantvillage_dataset.zip')
    print('Unzipping...')
    with zipfile.ZipFile('/content/plantvillage_dataset.zip') as z:
        z.extractall(LOCAL_DATA)

data_dir = pathlib.Path(LOCAL_DATA)
print('Dataset ready at', data_dir)

### Step 4: List every photo and its label
This goes through the 15 class folders and makes two lists:
- every photo's file location
- its label as a number from 0 to 14 (0 = first class alphabetically, and so on)

**Expected output:** 15 classes and a little over 20,000 photos, with a count for each class. Notice how uneven the classes are: some have thousands of photos, some only a couple of hundred. That's worth a sentence in your paper's dataset section.

In [ ]:
class_names = sorted(d.name for d in data_dir.iterdir() if d.is_dir())
paths, labels = [], []
for label, name in enumerate(class_names):
    for p in sorted((data_dir / name).iterdir()):
        if p.suffix.lower() in ('.jpg', '.jpeg', '.png'):
            paths.append(str(p))
            labels.append(label)
paths, labels = np.array(paths), np.array(labels)
NUM_CLASSES = len(class_names)

# shorter names for charts, e.g. 'Potato___Early_blight' -> 'Potato Early blight'
short_names = [n.replace('Pepper__bell___', 'Pepper ').replace('Tomato__Tomato_', 'Tomato ')
                .replace('___', ' ').replace('__', ' ').replace('_', ' ') for n in class_names]

print(NUM_CLASSES, 'classes,', len(paths), 'photos in total\n')
for label, name in enumerate(class_names):
    print(f'{name:45s} {np.sum(labels == label):5d}')

### Step 5: Split into train (70%) / validation (15%) / test (15%)
The flower tutorial only had two piles. Here you have all three:
- **train:** what the model studies
- **validation:** the quizzes during training
- **test:** the final exam, used once at the end of each run

The split is **stratified**: each class is split 70/15/15 on its own, so even a small class still shows up in all three piles. The exact split is saved to Drive (`results/data_split.csv`), so your paper can state precisely which images were used.

**Expected output:** roughly 14,400 train, 3,100 validation, 3,100 test.

In [ ]:
train_p, temp_p, train_y, temp_y = train_test_split(
    paths, labels, test_size=0.30, stratify=labels, random_state=SPLIT_SEED)
val_p, test_p, val_y, test_y = train_test_split(
    temp_p, temp_y, test_size=0.50, stratify=temp_y, random_state=SPLIT_SEED)

print('train:', len(train_p), ' validation:', len(val_p), ' test:', len(test_p))

split_table = pd.DataFrame({
    'file': np.concatenate([train_p, val_p, test_p]),
    'label': np.concatenate([train_y, val_y, test_y]),
    'split': ['train'] * len(train_p) + ['val'] * len(val_p) + ['test'] * len(test_p)})
split_table['file'] = split_table['file'].str.replace(LOCAL_DATA + '/', '', regex=False)
split_table.to_csv(f'{RESULTS_DIR}/data_split.csv', index=False)

### Step 6: Load and resize the photos
`load_photo` opens one photo, turns it into numbers, and resizes it to 224×224. These are the same "load → resize" steps as the flower tutorial, just written out by hand.

`cache()` keeps the resized photos in memory after the first pass. Later epochs don't have to reopen 14,000 files every time, which makes training much faster.

**Expected output:** nothing. The photos actually load the first time training starts.

In [ ]:
def load_photo(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(tf.clip_by_value(tf.round(image), 0, 255), tf.uint8)
    return image, label

def photo_dataset(p, y, cache=True):
    ds = tf.data.Dataset.from_tensor_slices((p, y)).map(load_photo, num_parallel_calls=AUTOTUNE)
    return ds.cache() if cache else ds

base_train = photo_dataset(train_p, train_y)
base_val = photo_dataset(val_p, val_y)
base_test = photo_dataset(test_p, test_y)

### Step 7: Augmentation and the colour-space switch
This is the heart of the experiment.

- **Augmentation** (flip, rotate, zoom) is the same "seatbelt" from the flower tutorial. It's only applied to training photos, and always *before* the colour conversion, so RGB and HSV models see exactly the same augmented photos.
- **`convert_colour_space`** is the one place where the two experiments differ. For `'rgb'` it does nothing. For `'hsv'` it converts each photo to Hue / Saturation / Value. Both end up on the same 0–255 scale, so everything after this point is identical.

*Change from the original roadmap:* the roadmap used OpenCV (`cv2`) for the HSV conversion. That version breaks when it's given a batch of photos at once. It also stores hue on a 0–179 scale while saturation and value use 0–255, so the three channels wouldn't be on equal footing. TensorFlow's built-in `rgb_to_hsv` avoids both problems.

**Expected output:** nothing.

In [ ]:
def convert_colour_space(images, colour_space):
    images = tf.clip_by_value(tf.cast(images, tf.float32), 0.0, 255.0)
    if colour_space == 'hsv':
        images = tf.image.rgb_to_hsv(images / 255.0) * 255.0
    return images

def make_dataset(base, colour_space, training=False, seed=None):
    ds = base
    if training:
        augment = tf.keras.Sequential([
            tf.keras.layers.RandomFlip('horizontal', seed=seed),
            tf.keras.layers.RandomRotation(0.1, seed=seed),
            tf.keras.layers.RandomZoom(0.1, seed=seed),
        ])
        ds = ds.shuffle(2000, seed=seed)
    ds = ds.batch(BATCH_SIZE)

    def augment_batch(images, labels):
        return augment(tf.cast(images, tf.float32), training=True), labels

    def colour_batch(images, labels):
        return convert_colour_space(images, colour_space), labels

    if training:
        ds = ds.map(augment_batch, num_parallel_calls=AUTOTUNE)
    ds = ds.map(colour_batch, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

### Step 8: See what the model will actually see
Top row: normal RGB photos. The three rows below: the same photos' **Hue**, **Saturation** and **Value** channels, shown in grey.

Notice that *Value* is basically a brightness map, while *Hue* shows colour identity. That separation is your whole hypothesis, in one picture. This figure is worth including in your paper's methodology section.

In [ ]:
preview = list(photo_dataset(val_p[:4], val_y[:4], cache=False).batch(4))[0]
images, image_labels = preview
hsv_images = convert_colour_space(images, 'hsv').numpy()

plt.figure(figsize=(12, 12))
for col in range(4):
    plt.subplot(4, 4, col + 1)
    plt.imshow(images[col].numpy())
    plt.title(short_names[image_labels[col]], fontsize=9)
    plt.axis('off')
    for row, channel in enumerate(['Hue', 'Saturation', 'Value']):
        plt.subplot(4, 4, (row + 1) * 4 + col + 1)
        plt.imshow(hsv_images[col, :, :, row], cmap='gray', vmin=0, vmax=255)
        plt.title(channel, fontsize=9)
        plt.axis('off')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/rgb_vs_hsv_channels.png', dpi=150)
plt.show()

### Step 9: A lighting-shift test set
PlantVillage photos were taken in even, controlled light, but your hypothesis is about *bad* lighting. Phase 7 tests that with your own photos, but 15–30 photos is a small sample. So every model is **also** scored on a "relit" copy of the test set. Each test photo gets:
- a random brightness change, from deep shade (0.4×) to harsh sun (1.6×)
- a contrast change
- a slight warm or cool colour tint, like direct sun vs. shade

The changes are random but **fixed**: every model gets the exact same relit photos, so the comparison stays fair.

*Fair-play note for your paper:* a pure brightness change leaves hue untouched by definition, which naturally favours HSV. The colour tint is included because real shade and sunlight also shift colour, which changes hue as well. So this test isn't rigged in HSV's favour.

**Expected output:** 4 test photos, before (top) and after relighting (bottom).

In [ ]:
def relight(index, photo_and_label):
    image, label = photo_and_label
    rnd = tf.random.stateless_uniform([3], seed=tf.stack([tf.cast(index, tf.int64), tf.constant(7, tf.int64)]))
    brightness = 0.4 + 1.2 * rnd[0]  # 0.4x (deep shade) to 1.6x (harsh sun)
    gamma = 0.7 + 0.8 * rnd[1]  # contrast change
    warmth = 0.85 + 0.30 * rnd[2]  # below 1 = cooler / bluer, above 1 = warmer / yellower
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.pow(image, gamma) * brightness
    image = image * tf.stack([warmth, 1.0, 2.0 - warmth])
    image = tf.clip_by_value(image, 0.0, 1.0) * 255.0
    return tf.cast(tf.round(image), tf.uint8), label

base_test_relit = base_test.enumerate().map(relight, num_parallel_calls=AUTOTUNE).cache()

before = list(photo_dataset(test_p[:4], test_y[:4], cache=False))
after = list(photo_dataset(test_p[:4], test_y[:4], cache=False).enumerate().map(relight))
plt.figure(figsize=(12, 6))
for i in range(4):
    plt.subplot(2, 4, i + 1); plt.imshow(before[i][0].numpy()); plt.title('original', fontsize=9); plt.axis('off')
    plt.subplot(2, 4, i + 5); plt.imshow(after[i][0].numpy()); plt.title('relit', fontsize=9); plt.axis('off')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/relit_examples.png', dpi=150)
plt.show()

# Phase 5: Train the RGB models

### Step 10: The two model designs
Your paper compares colour spaces, so the comparison has to be fair to both. There's a catch with transfer learning: **MobileNetV2 learned to see from millions of RGB photos.** Hand it HSV photos and it's a bit like reading with colour-swapped glasses. It might do worse just because the input is unfamiliar, not because HSV carries less information. A reviewer would spot this immediately.

So the experiment uses two designs, and each is trained on both colour spaces:
- **`mobilenet`:** transfer learning (MobileNetV2, pretrained, frozen), as in the roadmap. Fast and accurate, but RGB has a home advantage.
- **`scratch`:** a small CNN that learns from zero, like the flower-tutorial model. Slower, and usually a bit less accurate, but it has no colour-space bias, so it's the **fairer test of your hypothesis**.

If both designs agree, your conclusion is strong. If they disagree, that disagreement is itself an interesting result for your discussion section.

**Expected output:** nothing.

In [ ]:
def build_model(arch):
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1)(inputs)  # 0..255 -> -1..1

    if arch == 'mobilenet':
        base = tf.keras.applications.MobileNetV2(
            input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
        base.trainable = False  # keep the pretrained layers frozen
        x = base(x, training=False)
    elif arch == 'scratch':
        for filters in [32, 64, 128, 128]:
            x = tf.keras.layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
            x = tf.keras.layers.BatchNormalization()(x)
            x = tf.keras.layers.MaxPooling2D()(x)
    else:
        raise ValueError(f'unknown architecture: {arch}')

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES)(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer='adam',
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=['accuracy'])
    return model

### Step 11: The train-and-score routine
`run_experiment` does one complete run: build a fresh model → train it → score it on the test set and the relit test set → save everything to Drive. For each run it saves, inside `results/`:
- a row in `results.csv`: all the numbers for your results table
- `curves.png`: the same accuracy/loss plots you made in the flower tutorial
- `confusion_test.png`: which classes get confused with which
- `model.keras`: the trained model itself, which you'll need in Phase 7 for your own photos

It gives two scores per test set:
- **Accuracy:** % of photos it got right.
- **Macro F1:** a score that treats every class equally, so a model can't look good just by being right about the big classes. Your classes are very uneven, so report both.

If a run is already in `results.csv`, it's skipped. That's what makes it safe to re-run after a disconnect.

**Expected output:** nothing yet.

In [ ]:
def score(model, base, true_labels, colour_space):
    logits = model.predict(make_dataset(base, colour_space), verbose=0)
    predicted = np.argmax(logits, axis=1)
    accuracy = float(np.mean(predicted == true_labels))
    macro_f1 = float(f1_score(true_labels, predicted, average='macro', zero_division=0))
    return accuracy, macro_f1, confusion_matrix(true_labels, predicted, labels=range(NUM_CLASSES))

def save_plots(history, cm, name, run_dir):
    h = history.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h['accuracy'], label='Training'); axes[0].plot(h['val_accuracy'], label='Validation')
    axes[0].set_title('Accuracy'); axes[0].set_xlabel('epoch'); axes[0].legend()
    axes[1].plot(h['loss'], label='Training'); axes[1].plot(h['val_loss'], label='Validation')
    axes[1].set_title('Loss'); axes[1].set_xlabel('epoch'); axes[1].legend()
    fig.suptitle(name); fig.tight_layout()
    fig.savefig(f'{run_dir}/curves.png', dpi=120)
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 10))
    ConfusionMatrixDisplay(cm, display_labels=short_names).plot(ax=ax, xticks_rotation=90, colorbar=False)
    ax.set_title(f'{name}: confusion matrix (test set)')
    fig.tight_layout()
    fig.savefig(f'{run_dir}/confusion_test.png', dpi=120)
    plt.show()

def already_done(arch, colour_space, seed):
    if not os.path.exists(RESULTS_CSV):
        return False
    done = pd.read_csv(RESULTS_CSV)
    return bool(((done.arch == arch) & (done.colour_space == colour_space) & (done.seed == seed)).any())

def run_experiment(arch, colour_space, seed):
    name = f'{arch}_{colour_space}_seed{seed}'
    if already_done(arch, colour_space, seed):
        print(f'{name}: already finished, skipping')
        return
    print(f'\n=== {name} ===')
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)

    model = build_model(arch)
    start = time.time()
    history = model.fit(make_dataset(base_train, colour_space, training=True, seed=seed),
                        validation_data=make_dataset(base_val, colour_space),
                        epochs=EPOCHS[arch], verbose=2,
                        shuffle=False)  # shuffling already happens inside make_dataset
    minutes = (time.time() - start) / 60

    test_acc, test_f1, test_cm = score(model, base_test, test_y, colour_space)
    relit_acc, relit_f1, relit_cm = score(model, base_test_relit, test_y, colour_space)

    run_dir = f'{RESULTS_DIR}/{name}'
    os.makedirs(run_dir, exist_ok=True)
    model.save(f'{run_dir}/model.keras')
    pd.DataFrame(history.history).to_csv(f'{run_dir}/history.csv', index=False)
    np.save(f'{run_dir}/confusion_test.npy', test_cm)
    np.save(f'{run_dir}/confusion_relit.npy', relit_cm)
    save_plots(history, test_cm, name, run_dir)

    row = pd.DataFrame([{
        'arch': arch, 'colour_space': colour_space, 'seed': seed, 'epochs': EPOCHS[arch],
        'train_acc': round(history.history['accuracy'][-1], 4),
        'val_acc': round(history.history['val_accuracy'][-1], 4),
        'test_acc': round(test_acc, 4), 'test_macro_f1': round(test_f1, 4),
        'relit_test_acc': round(relit_acc, 4), 'relit_macro_f1': round(relit_f1, 4),
        'train_minutes': round(minutes, 1)}])
    row.to_csv(RESULTS_CSV, mode='a', header=not os.path.exists(RESULTS_CSV), index=False)
    print(f'{name}: test accuracy {test_acc:.3f} | relit test accuracy {relit_acc:.3f} | {minutes:.1f} min')

### Step 12: Run the RGB experiments
This trains **6 models**: 2 designs × 3 seeds, all on RGB. Rough timing on Colab's free GPU is about 5–10 minutes per MobileNet run and 10–20 minutes per scratch run, so **roughly 1–2 hours** in total.

- You don't need to watch it, but keep the tab open. Colab can disconnect idle tabs.
- If it disconnects, or Colab says you've hit your GPU limit for the day, stop. Next time, run from the top and it carries on from where it stopped.
- Each epoch prints a line, like in the flower tutorial. Watch the same thing as before: training and validation accuracy should rise together.

**Expected output:** training lines, two charts per run, and a summary line per run.

In [ ]:
for arch in ARCHITECTURES:
    for seed in RUN_SEEDS:
        run_experiment(arch, 'rgb', seed)

# Phase 6: Train the HSV models

Exactly the same routine: same designs, same seeds, same split, same epochs, same augmentation. The only thing that changes is the word `'hsv'`. That single change is what makes the comparison scientifically clean.

Same timing as Phase 5, roughly 1–2 hours.

In [ ]:
for arch in ARCHITECTURES:
    for seed in RUN_SEEDS:
        run_experiment(arch, 'hsv', seed)

### Quick look at the results
This is just a first peek. The real analysis (which classes each model confuses, and why) is Phase 8.

The table shows the **average** (`mean`) and **spread** (`std`) across the 3 seeds, for each design and colour space. The spread is how much the 3 runs disagreed with each other. If the gap between RGB and HSV is smaller than the spread, the difference probably isn't real.

In [ ]:
results = pd.read_csv(RESULTS_CSV)
summary = (results.groupby(['arch', 'colour_space'])
           [['test_acc', 'test_macro_f1', 'relit_test_acc', 'relit_macro_f1']]
           .agg(['mean', 'std']).round(4))
summary